# 面试问题：类别极不平衡时应该怎样训练和评估模型？

**一句话回答**：先确认不平衡是否符合真实先验以及漏报/误报成本；切分保持 group/time 合同。训练可用 class weight、受控重采样或 focal loss，但它们会改变概率含义；评估不能只看 accuracy，要报告 PR 曲线、precision/recall、成本、分组表现和校准；业务阈值只能在 validation 选择，再在封存 test 验证。

本 Notebook 用 NumPy/PyTorch 基础算子实现混淆矩阵、加权 BCE、Focal Loss、加权采样、线性模型和成本阈值。

In [ ]:
import hashlib, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED92=9201; rng92=np.random.default_rng(SEED92); torch.manual_seed(SEED92)
n92=3000; x92=rng92.normal(size=(n92,3)).astype(np.float32); logits_true92=1.8*x92[:,0]-1.1*x92[:,1]+.5*x92[:,2]-3.2; p_true92=1/(1+np.exp(-logits_true92)); y92=(rng92.random(n92)<p_true92).astype(np.float32)
assert x92.shape==(3000,3) and .03<y92.mean()<.15
assert np.isfinite(p_true92).all() and np.all((p_true92>0)&(p_true92<1))
assert torch.__version__

## 1. Accuracy 反例与混淆矩阵

若阳性率 6%，全预测负类已有 94% accuracy，但 recall=0。先从 TP/FP/FN/TN 定义 precision、recall、specificity、F1；没有预测阳性时 precision 的约定要明确，本例返回 0。

指标必须报告绝对样本数，因为 1% false-positive rate 在百万流量下仍是大量人工审核。

In [ ]:
def confusion92(y,score,threshold):
    y=np.asarray(y,int); pred=np.asarray(score)>=threshold; tp=int(np.sum((y==1)&pred)); fp=int(np.sum((y==0)&pred)); fn=int(np.sum((y==1)&~pred)); tn=int(np.sum((y==0)&~pred)); return tp,fp,fn,tn
def metrics92(cm):
    tp,fp,fn,tn=cm; div=lambda a,b:a/b if b else 0.; precision=div(tp,tp+fp); recall=div(tp,tp+fn); return {"accuracy":div(tp+tn,sum(cm)),"precision":precision,"recall":recall,"specificity":div(tn,tn+fp),"f1":div(2*precision*recall,precision+recall)}
all_negative92=metrics92(confusion92(y92,np.zeros(n92),.5))
assert all_negative92["accuracy"]>.85 and all_negative92["recall"]==0
assert all_negative92["precision"]==0 and all_negative92["f1"]==0
assert sum(confusion92(y92,p_true92,.5))==n92

## 2. 加权 BCE 的含义

`pos_weight=N_negative/N_positive` 放大阳性梯度，使模型更关注漏报；它不是凭空增加信息，也不保证概率仍校准。PyTorch BCEWithLogits 的稳定形式避免先 sigmoid 再 log 的溢出。

下面手写稳定 binary loss：`max(z,0)-z*y+log(1+exp(-|z|))`，再只给正例项乘权重。

In [ ]:
def weighted_bce92(logits,targets,pos_weight=1.):
    z=np.asarray(logits,float); y=np.asarray(targets,float); base=np.maximum(z,0)-z*y+np.log1p(np.exp(-np.abs(z))); weights=np.where(y==1,pos_weight,1.); return base*weights
probe_z92=np.array([-100.,-2.,0.,2.,100.]); probe_y92=np.array([0,1,0,1,1]); manual92=weighted_bce92(probe_z92,probe_y92,3)
torch92=F.binary_cross_entropy_with_logits(torch.tensor(probe_z92),torch.tensor(probe_y92,dtype=torch.float64),pos_weight=torch.tensor(3.,dtype=torch.float64),reduction="none").numpy()
assert np.allclose(manual92,torch92,atol=1e-10)
assert np.isfinite(manual92).all() and manual92[1]>weighted_bce92([-2],[1],1)[0]
pos_weight92=float((y92==0).sum()/(y92==1).sum()); assert pos_weight92>5

## 3. Focal Loss 从公式实现

`FL=-alpha_t(1-p_t)^gamma log(p_t)`；易分类样本 `p_t` 接近 1，权重快速变小，训练集中在困难样本。gamma=0 退化为带 alpha 的 BCE。困难样本可能是标注错误，因此 focal 需要和数据清洗、loss 分布监控一起使用。

实现直接从 logits 调用稳定 BCE，再计算 `p_t=exp(-BCE)`。

In [ ]:
def focal_loss92(logits,targets,alpha=.25,gamma=2.):
    logits=torch.as_tensor(logits,dtype=torch.float32); targets=torch.as_tensor(targets,dtype=torch.float32); bce=F.binary_cross_entropy_with_logits(logits,targets,reduction="none"); pt=torch.exp(-bce); alpha_t=torch.where(targets==1,torch.tensor(alpha),torch.tensor(1-alpha)); return alpha_t*(1-pt).pow(gamma)*bce
easy92=focal_loss92([6.,-6.],[1.,0.]); hard92=focal_loss92([0.,0.],[1.,0.])
assert hard92.mean()>easy92.mean()*100
assert torch.allclose(focal_loss92([1.,-1.],[1.,0.],.5,0),.5*F.binary_cross_entropy_with_logits(torch.tensor([1.,-1.]),torch.tensor([1.,0.]),reduction="none"))
assert torch.isfinite(focal_loss92([100.,-100.],[1.,0.])).all()

## 4. 重采样与重复过拟合

weighted sampler 按类别反频率抽样，让 batch 更平衡；但少数类样本会重复出现，标注噪声也被放大。验证/test 绝不能重采样，指标必须在真实先验上计算。按 group 抽样时还需避免一个大 group 主导。

抽样后的 loss 若按新分布训练，概率要通过真实先验数据重新校准。

In [ ]:
class_count92=np.bincount(y92.astype(int)); sample_weights92=np.where(y92==1,1/class_count92[1],1/class_count92[0]); sample_weights92=sample_weights92/sample_weights92.sum(); sampled_idx92=rng92.choice(n92,size=n92,replace=True,p=sample_weights92)
sampled_rate92=float(y92[sampled_idx92].mean())
assert .45<sampled_rate92<.55
assert not math.isclose(sampled_rate92,float(y92.mean()),abs_tol=.2)
assert len(np.unique(sampled_idx92))<n92

## 5. 手写小线性模型比较普通/加权训练

同一训练切分、初始化和优化器下比较 BCE 与 pos-weight BCE。加权模型通常提高固定 0.5 阈值 recall，但 precision/校准可能下降；这说明“训练损失选择”和“业务阈值选择”是两个问题。

模型只是一层 `nn.Linear`，没有调用现成分类器。

In [ ]:
train92=np.arange(0,2000); val92=np.arange(2000,2500); test92=np.arange(2500,3000)
class Linear92(nn.Module):
    def __init__(self): super().__init__(); self.linear=nn.Linear(3,1)
    def forward(self,x): return self.linear(x).squeeze(-1)
def train92_model(weighted):
    torch.manual_seed(92); m=Linear92(); opt=torch.optim.SGD(m.parameters(),lr=.15); xt=torch.tensor(x92[train92]); yt=torch.tensor(y92[train92])
    pw=torch.tensor(float((yt==0).sum()/(yt==1).sum()))
    for _ in range(80):
        loss=F.binary_cross_entropy_with_logits(m(xt),yt,pos_weight=pw if weighted else None); opt.zero_grad(); loss.backward(); opt.step()
    return m
plain92=train92_model(False); weighted92=train92_model(True)
with torch.no_grad(): plain_score92=torch.sigmoid(plain92(torch.tensor(x92))).numpy(); weighted_score92=torch.sigmoid(weighted92(torch.tensor(x92))).numpy()
pm92=metrics92(confusion92(y92[test92],plain_score92[test92],.5)); wm92=metrics92(confusion92(y92[test92],weighted_score92[test92],.5))
assert wm92["recall"]>pm92["recall"]
assert weighted_score92[y92==1].mean()>weighted_score92[y92==0].mean()
assert all(torch.isfinite(p).all() for p in weighted92.parameters())

## 6. 按业务成本在 validation 选阈值

假设漏报成本 8、误报成本 1，遍历 validation 的候选阈值，最小化 `8*FN+FP`。阈值属于模型发布制品，不能在 test 上反复挑。若每日人工审核容量有限，还需加入 predicted-positive 数量约束。

测试集只用于报告选定阈值表现，不再调整。

In [ ]:
def choose_threshold92(y,score,fn_cost=8,fp_cost=1):
    candidates=np.unique(np.r_[0.,score,1.]); rows=[]
    for t in candidates:
        tp,fp,fn,tn=confusion92(y,score,t); rows.append((fn_cost*fn+fp_cost*fp,t,(tp,fp,fn,tn)))
    return min(rows,key=lambda x:(x[0],x[1]))
val_choice92=choose_threshold92(y92[val92],weighted_score92[val92]); threshold92=val_choice92[1]; test_cm92=confusion92(y92[test92],weighted_score92[test92],threshold92); test_metrics92=metrics92(test_cm92)
assert 0<=threshold92<=1 and sum(test_cm92)==len(test92)
assert test_metrics92["recall"]>.65
assert val_choice92[0]<=8*int(y92[val92].sum())

## 7. 切片、先验变化与校准

总体少数类可能来自某些 region/device，必须按组报告 recall/precision 和样本数。部署先验变化会改变 precision，即使 ROC 不变；class weight/重采样也会改变输出概率，不能把 0.5 当天然业务阈值。

训练、validation、test 的阳性率应记录；若差异来自采样而非真实时间变化，需要还原权重。

In [ ]:
groups92=np.where(x92[:,2]>0,"g1","g0"); slice_metrics92={g:metrics92(confusion92(y92[test92][groups92[test92]==g],weighted_score92[test92][groups92[test92]==g],threshold92)) for g in ("g0","g1")}
assert set(slice_metrics92)=={"g0","g1"}
assert all(0<=m["recall"]<=1 and 0<=m["precision"]<=1 for m in slice_metrics92.values())
assert all(np.sum(groups92[test92]==g)>150 for g in slice_metrics92)

## 8. 发布与监控合同

manifest 绑定训练先验、loss/weight/gamma、sampling、validation 阈值、成本矩阵、split 和校准版本。线上监控预测阳性率、precision/recall（标签回流后）、人工审核容量、分组漏报和 prior shift；阈值灰度应与模型版本一起发布。

只汇报 weighted loss 下降不能证明业务收益，最终要回到真实分布和成本。

In [ ]:
state92=b"".join(p.detach().numpy().tobytes() for p in weighted92.parameters()); manifest92={"schema":1,"model":"linear-weighted-bce","state_sha256":hashlib.sha256(state92).hexdigest(),"train_positive_rate":float(y92[train92].mean()),"pos_weight":pos_weight92,"threshold":float(threshold92),"fn_cost":8,"fp_cost":1}
digest92=hashlib.sha256(json.dumps(manifest92,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()
assert len(digest92)==64 and manifest92["fn_cost"]>manifest92["fp_cost"]
assert math.isclose(manifest92["threshold"],threshold92)
assert manifest92["train_positive_rate"]<.2
print({"positive_rate":round(float(y92.mean()),3),"plain":pm92,"weighted":wm92,"threshold":round(threshold92,3)})

## 9. 面试收束、参考与练习

回答闭环：真实先验/成本 → accuracy 反例 → weighted BCE/focal/sampling → 真实分布评估 → validation 选阈值 → 分组与 prior shift → 发布/标签回流。没有一种“不平衡专用 loss”能替代数据质量和业务决策。

练习：实现 PR-AUC；给阈值加入审核容量；比较 focal gamma；对重采样模型做先验校正；模拟正例标签漏标。

参考：[Focal Loss 原论文](https://arxiv.org/abs/1708.02002)、[Precision-Recall 与 ROC 的关系](https://dl.acm.org/doi/10.1145/1143844.1143874)、[PyTorch BCEWithLogitsLoss 公式](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)。